In [0]:
import pyspark.sql.functions as sf

SCHEMA = dbutils.widgets.get("SCHEMA")

TABLE_BRONZE_PRODUCTS_INFO = SCHEMA + "." + dbutils.widgets.get("TABLE_BRONZE_PRODUCTS_INFO")
TABLE_BRONZE_REVIEWS = SCHEMA + "." + dbutils.widgets.get("TABLE_BRONZE_REVIEWS")
TABLE_SILVER_PRODUCTS_RECOMMENDATIONS = SCHEMA + "." + dbutils.widgets.get("TABLE_SILVER_PRODUCTS_RECOMMENDATIONS")

In [0]:
df_bronze_products_info = spark.read.table(TABLE_BRONZE_PRODUCTS_INFO)
df_bronze_reviews = spark.read.table(TABLE_BRONZE_REVIEWS)

In [0]:
df_silver_products_recommendations = (
  df_bronze_products_info
  .select("product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights", )
  .join(
    df_bronze_reviews.select(
      "product_id", "rating", "is_recommended", "review_text", "price_usd"
    ),
    on="product_id",
    how="inner"
  )
  .withColumn("id", sf.monotonically_increasing_id())
)

(
    df_silver_products_recommendations
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(TABLE_SILVER_PRODUCTS_RECOMMENDATIONS)
)